In [44]:
### Import des modules
%load_ext autoreload
%autoreload 2

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import utils
from sklearn.ensemble import RandomForestClassifier
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression

pd.set_option('display.max_rows', 500)
pd.set_option('display.max_columns', 500)
pd.set_option('display.width', 1000)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [45]:
data = pd.read_csv('data/rafined/employees.csv', sep=',')

In [46]:
# Plus un employé a d'ancienneté, moins il va avoir tendance à partir
data['ratio_anciennete_total_experence'] = np.where(
    data['annee_experience_totale'] > 0,
    data['annees_dans_l_entreprise'] / data['annee_experience_totale'],
    data['annees_dans_l_entreprise']
)

# Plus un employé évolue au sein de l'entreprise, moins il va avoir tendance à partir
data['ratio_poste_actuel_anciennete'] = np.where(
    data['annees_dans_l_entreprise'] > 0,
    data['annees_dans_le_poste_actuel'] / data['annees_dans_l_entreprise'],
    data['annees_dans_le_poste_actuel']
)

# Indicateur de pénibilité de déplacement distance * frequence (1=jamais, 1.5=occasionnel, 2=frequent)
data['score_penibilite_transport'] = data['distance_domicile_travail'] * data['frequence_deplacement']

# Plus le score bien-être est élevée moins un salarié va avoir tendance a quitter l'entreprise
colonnes_satisfaction = [
    'satisfaction_employee_environnement',
    'satisfaction_employee_nature_travail',
    'satisfaction_employee_equipe',
    'satisfaction_employee_equilibre_pro_perso'
]
data['score_bien_etre'] = data[colonnes_satisfaction].mean(axis=1)

# Plus le score de peformance est élevée plus un salarié va rester
colonnes_evaluation = [
    'note_evaluation_precedente',
    'note_evaluation_actuelle'
]

data['score_performance'] = data[colonnes_evaluation].mean(axis=1)

# Si l'évalusation est à la baisse, baisse des résultats donc salarié moins satisfait ?
data['evolution_performance'] = data['note_evaluation_actuelle'] - data['note_evaluation_precedente']

#
data['evolution_hierarchie_score'] = np.where(
    data['annee_experience_totale'] > 0,
    data['niveau_hierarchique_poste'] / data['annee_experience_totale'],
    data['niveau_hierarchique_poste']
)

# Plus un employé s'en va rapidement de ses anciennes entreprises, plus il aura tendance à partir rapidement ?
data['annee_par_experience'] = np.where(
    data['nombre_experiences_precedentes'] > 0,
    data['annee_experience_totale'] / data['nombre_experiences_precedentes'],
    data['annee_experience_totale']
)

# Création d'une colonne pour savoir comment se positione le salarié en salaire par rapport à ses collègues au même poste
data['quartile_salaire_par_poste'] = data.groupby('poste')['revenu_mensuel'].transform(
    lambda x: pd.qcut(x, q=4, labels=[1, 2, 3, 4], duplicates='drop')
)

In [47]:
data = data[['revenu_mensuel',
'annee_par_experience',
'heure_supplementaires',
'age',
'statut_marital',
'score_penibilite_transport',
'annee_experience_totale',
'annees_dans_l_entreprise',
'evolution_hierarchie_score',
'ratio_poste_actuel_anciennete',
'score_bien_etre',
'nombre_participation_pee',
'ratio_anciennete_total_experence',
'annees_dans_le_poste_actuel',
'annes_sous_responsable_actuel',
'nombre_experiences_precedentes',
'niveau_hierarchique_poste',
'satisfaction_employee_environnement',
'nb_formations_suivies',
'satisfaction_employee_nature_travail',
'annees_depuis_la_derniere_promotion',
'satisfaction_employee_equipe',
'quartile_salaire_par_poste',
'score_performance',
'niveau_education',
'satisfaction_employee_equilibre_pro_perso',
'a_quitte_l_entreprise',
'evolution_performance',]]

In [48]:
train_data = utils.split_train_data(data, 'a_quitte_l_entreprise')

In [49]:
from sklearn.preprocessing import RobustScaler, MinMaxScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer

preprocessor = ColumnTransformer(
    transformers=[
        ('robust_scaler', RobustScaler(), ['revenu_mensuel']),
        ('minmax_scaler', MinMaxScaler(), ['age']),
        ('encoder', OneHotEncoder(), ['statut_marital']),
    ],
    remainder='passthrough'
)

In [50]:
from sklearn.model_selection import FixedThresholdClassifier
from technova_features import TechNovaFeatureEngineering
from sklearn.pipeline import Pipeline

random_forest_model = RandomForestClassifier(random_state=42, class_weight='balanced')

threshold_model = FixedThresholdClassifier(
    estimator=random_forest_model,
    threshold=0.3,
    response_method="predict_proba"
)

pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', threshold_model),
])
utils.benchmark(pipeline, train_data)

CrossValidation Results : 
{'fit_time': array([0.11363387, 0.10992098, 0.10652804, 0.10881305, 0.10594416]), 'score_time': array([0.01015401, 0.00876999, 0.00850296, 0.008497  , 0.00849891]), 'test_recall': array([0.775     , 0.74358974, 0.69230769, 0.75      , 0.85      ]), 'test_f1': array([0.50406504, 0.52252252, 0.44628099, 0.48780488, 0.5483871 ])}
Recall moyen : 0.7621794871794871
F1-Score moyen : 0.501812105946288
Training Résults : 
Recall moyen : 0.6153846153846154
F1-Score moyen : 0.375
[[190  65]
 [ 15  24]]
